# تحليل حوادث المرور في المملكة العربية السعودية (١٤٣٧–١٤٣٩ هـ)


## ٢. مقدمة

يحلّل هذا المشروع بيانات حوادث المرور في المملكة العربية السعودية على مدى ثلاث سنوات هجرية (١٤٣٧ و١٤٣٨ و١٤٣٩). البيانات تشمل سجلات شهرية لـ **١٦ مدينة**، وتحتوي على تفاصيل ديموغرافية مثل الجنس والفئة العمرية والجنسية وموقع الحادث.

**أهداف التحليل:**
- استكشاف الأنماط الزمنية والجغرافية للحوادث
- تحديد المدن والفئات الأكثر عرضة للخطر
- بناء نماذج ذكاء اصطناعي لدعم قرارات السلامة المرورية
- تجميع المدن في مجموعات متشابهة لتوجيه السياسات الأمنية


## ٣. استيراد المكتبات


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# مكتبات التعلم الآلي
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import (classification_report, confusion_matrix,
                             roc_auc_score, roc_curve, ConfusionMatrixDisplay)
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA

sns.set_theme(style='whitegrid', font_scale=1.1)
plt.rcParams.update({'figure.facecolor': '#F8F9FA', 'axes.facecolor': '#F8F9FA'})

print('Libraries loaded ✓')


## ٤. تحميل البيانات

ملفات **المصابين** تستخدم الفاصلة `,` — ملفات **المتوفين** تستخدم الفاصلة المنقوطة `;`

In [ ]:
df1  = pd.read_csv("Injured in Accidents 1437 H.csv", engine="python", encoding="utf-8-sig", header=None)
df2  = pd.read_csv("Injured in Accidents 1438 H.csv", engine="python", encoding="utf-8-sig", header=None)
df3  = pd.read_csv("Injured in Accidents 1439 H.csv", engine="python", encoding="utf-8-sig", header=None)

df11 = pd.read_csv("Dead in Accidents 1437 H.csv", engine="python", encoding="utf-8-sig", header=None, sep=";")
df22 = pd.read_csv("Dead in Accidents 1438 H.csv", engine="python", encoding="utf-8-sig", header=None, sep=";")
df33 = pd.read_csv("Dead in Accidents 1439 H.csv", engine="python", encoding="utf-8-sig", header=None, sep=";")

print("All 6 files loaded ✓")


## ٥. نظرة أولية على البيانات

| الأمر | ما يعرضه |
|---|---|
| `.head()` | أول صفوف من البيانات |
| `.shape` | عدد الصفوف والأعمدة |
| `.info()` | أنواع البيانات وعدد القيم الفارغة |
| `.describe()` | إحصاءات وصفية (متوسط، أدنى، أعلى...) |


In [ ]:
print("Shape:", df1.shape)
display(df1.head(5))


## ٦. دالة تنظيف البيانات

### ⚠️ مشكلة في البيانات — وكيف حللناها

عند الفحص وجدنا أن ملفات **المصابين** تُنتج **١٨٩ صفاً** بدلاً من **١٩٢** (١٦ مدينة × ١٢ شهراً).

**السبب:** مدينة **القريات** في ملفات المصابين تحتوي على **تهجئات عربية مختلفة** لثلاثة أشهر:

| التهجئة في الملف | الشهر الصحيح |
|---|---|
| `مــحــر م` (بمسافات إضافية) | محرم |
| `صــــفـر` (بمسافات إضافية) | صفر |
| `ر جـــب` (بمسافات إضافية) | رجب |

**الحل:** إضافة هذه التهجئات البديلة إلى قاموس الشهور في دالة التنظيف.

بعد الإصلاح: **١٩٢ صفاً لكل ملف** ✓

In [ ]:
def clean_accidents_data(df, year, data_type):

    # --- الخطوة ١: تحديد أسماء الأعمدة ---
    # الصف الثاني في الملف الخام هو أسماء الأعمدة
    df.columns = df.iloc[1]
    df = df.iloc[2:].reset_index(drop=True)  # إزالة صفي الرأس

    # --- الخطوة ٢: إعادة تسمية الأعمدة بالإنجليزية ---
    columns = [
        'month', 'male', 'female', 'total_gender',
        'inside_city', 'outside_city', 'total_location',
        'under_18', 'age_18_30', 'age_30_40', 'age_40_50', 'age_50_plus', 'total_age',
        'saudi', 'non_saudi', 'total_nationality'
    ]
    df.columns = columns

    # --- الخطوة ٣: حذف صفوف الترويسات المكررة ---
    # كل مدينة في الملف تبدأ بصفوف تحتوي على كلمة 'عدد' — نحذفها
    df = df[~df.apply(lambda row: row.astype(str).str.contains('عدد').any(), axis=1)]

    # --- الخطوة ٤: تحويل أسماء الشهور إلى أرقام ---
    # الإصلاح: أضفنا التهجئات البديلة لمدينة القريات
    month_map = {
        'محرم': 1,      'صفر': 2,         'ربيع أول': 3,   'ربيع ثانى': 4,
        'جمادى أول': 5, 'جمادى ثانى': 6,  'رجب': 7,        'شعبان': 8,
        'رمضان': 9,     'شوال': 10,        'ذى القعدة': 11, 'ذى الحجة': 12,
        # تهجئات بديلة موجودة في ملفات المصابين (القريات فقط)
        'مــحــر م': 1,  'صــــفـر': 2,  'ر جـــب': 7
    }
    df['month'] = df['month'].map(month_map)

    # --- الخطوة ٥: حذف الصفوف الفارغة ---
    # الصفوف التي لم تُطابق أي شهر (مجاميع، فراغات) تصبح NaN فنحذفها
    df = df.dropna()
    df['month'] = df['month'].astype(int)

    # --- الخطوة ٦: حذف صفوف المجاميع السنوية ---
    # آخر 12 صف هي مجاميع كل المدن معاً — لا نحتاجها
    df = df.iloc[:-12]

    # --- الخطوة ٧: إضافة عمود السنة ونوع البيانات ---
    df['year'] = year
    if data_type == 'injured':
        df['injured'] = 1
        df['dead'] = 0
    elif data_type == 'dead':
        df['injured'] = 0
        df['dead'] = 1

    # --- الخطوة ٨: حذف أعمدة المجاميع ---
    # لا نحتاج أعمدة 'total_*' لأننا سنحسبها بأنفسنا عند الحاجة
    columns_to_drop = [col for col in df.columns if 'total' in col]
    df = df.drop(columns=columns_to_drop)

    # --- الخطوة ٩: إضافة عمود المدينة ---
    # البيانات مرتبة بالتسلسل: 12 شهراً لكل مدينة
    cities = [
        'Riyadh', 'Jeddah', 'Madinah', 'Al-Sharqiyah', 'Northern Borders',
        'Tabuk', 'Al-Jouf', 'Hail', 'Najran', 'Al-Qassim', 'Al-Baha',
        'Asir', 'Jazan', 'Taif', 'Makkah', 'Al-Qurayyat'
    ]
    city_column_data = []
    for city in cities:
        city_column_data.extend([city] * 12)
    city_column_data = city_column_data[:len(df)]
    df['city'] = city_column_data

    return df


# --- تطبيق دالة التنظيف على جميع الملفات ---
injured_1437 = clean_accidents_data(df1,  1437, 'injured')
injured_1438 = clean_accidents_data(df2,  1438, 'injured')
injured_1439 = clean_accidents_data(df3,  1439, 'injured')

dead_1437 = clean_accidents_data(df11, 1437, 'dead')
dead_1438 = clean_accidents_data(df22, 1438, 'dead')
dead_1439 = clean_accidents_data(df33, 1439, 'dead')

# --- التحقق من تساوي الصفوف بعد الإصلاح ---
print('=== Row Count Verification ===')
for yr, i_df, d_df in [(1437, injured_1437, dead_1437),
                        (1438, injured_1438, dead_1438),
                        (1439, injured_1439, dead_1439)]:
    match = '✓ MATCH' if len(i_df) == len(d_df) else '✗ MISMATCH'
    print(f'{yr}  Injured: {len(i_df)}  Dead: {len(d_df)}  {match}')

# --- دمج كل الجداول في جدول واحد ---
df = pd.concat([
    injured_1437, injured_1438, injured_1439,
    dead_1437,    dead_1438,    dead_1439
], ignore_index=True)

# الأعمدة الرقمية
NUM_COLS = ['male', 'female', 'inside_city', 'outside_city',
            'under_18', 'age_18_30', 'age_30_40', 'age_40_50',
            'age_50_plus', 'saudi', 'non_saudi']
df[NUM_COLS] = df[NUM_COLS].apply(pd.to_numeric, errors='coerce')

# فصل المصابين عن المتوفين للتحليل المنفصل
inj = df[df['injured'] == 1].copy()
ded = df[df['dead']    == 1].copy()

# أسماء الشهور الإنجليزية للرسوم البيانية
MONTH_LABELS = [
    'Muharram', 'Safar', "Rabi' I", "Rabi' II",
    'Jumada I', 'Jumada II', 'Rajab', "Sha'ban",
    'Ramadan', 'Shawwal', "Dhul-Qa'da", 'Dhul-Hijja'
]

print(f'\nCombined DataFrame: {df.shape}')
print(f'Injured rows     : {len(inj)}  (expected 192×3 = 576... checking below)')
print(f'Dead rows        : {len(ded)}')
display(df.head())
print('\nInfo:')
df.info()
print('\nDescribe:')
display(df[NUM_COLS].describe().round(1))


## ٧. المخططات البيانية والاستنتاجات


### ٧.١ إجمالي الحوادث عبر السنوات — Line Chart

يعرض إجمالي **المصابين** و**المتوفين** لكل سنة هجرية. الهدف: هل الحوادث في ازدياد أم تناقص؟

In [ ]:
# --- إجمالي المصابين والمتوفين لكل سنة ---
yr_inj = inj.groupby('year')[NUM_COLS].sum().sum(axis=1)  # جمع كل الأعمدة الرقمية لكل سنة
yr_ded = ded.groupby('year')[NUM_COLS].sum().sum(axis=1)

fig, ax = plt.subplots(figsize=(9, 5))

# رسم خطي مع نقاط مميزة
ax.plot(yr_inj.index, yr_inj.values, 'o-', color='#2980B9', lw=2.5,
        ms=9, label='Injured', markerfacecolor='white', markeredgewidth=2.5)
ax.plot(yr_ded.index, yr_ded.values, 's--', color='#C0392B', lw=2.5,
        ms=9, label='Deaths', markerfacecolor='white', markeredgewidth=2.5)

# إضافة الأرقام فوق/أسفل كل نقطة
for x, yi, yd in zip(yr_inj.index, yr_inj.values, yr_ded.values):
    ax.annotate(f'{int(yi):,}', (x, yi), textcoords='offset points',
                xytext=(0, 12), ha='center', fontsize=11, color='#2980B9', fontweight='bold')
    ax.annotate(f'{int(yd):,}', (x, yd), textcoords='offset points',
                xytext=(0, -18), ha='center', fontsize=11, color='#C0392B', fontweight='bold')

ax.fill_between(yr_inj.index, yr_inj.values, alpha=0.08, color='#2980B9')
ax.fill_between(yr_ded.index, yr_ded.values, alpha=0.08, color='#C0392B')
ax.set_xticks(yr_inj.index)
ax.set_xticklabels(['1437 H', '1438 H', '1439 H'], fontsize=12)
ax.set_title('Total Accidents Over Time (1437–1439 H)', fontsize=14, fontweight='bold', pad=14)
ax.set_ylabel('Total Cases')
ax.legend(fontsize=11)
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.show()


#### 💡 الاستنتاجات

- **انخفاض تدريجي** في أعداد المصابين من ١٤٣٧ إلى ١٤٣٩، مما يدل على تحسّن في السلامة المرورية.
- رغم انخفاض المصابين، أعداد المتوفين لم تنخفض بنفس النسبة — مما يعني أن الحوادث أصبحت **أقل عدداً لكن أشد خطورة**.


### ٧.٢ خريطة حرارية للحوادث الشهرية — Heatmap

اللون الأحمر الداكن = أعلى عدد حوادث في ذلك الشهر.

In [ ]:
# --- تحويل البيانات لجدول شهر × سنة ---
pivot = inj.groupby(['year', 'month'])[NUM_COLS].sum().sum(axis=1).unstack()
pivot.columns = MONTH_LABELS  # استبدال أرقام الأشهر بأسمائها

fig, ax = plt.subplots(figsize=(15, 3.8))
sns.heatmap(pivot, annot=True, fmt='.0f', cmap='YlOrRd',
            linewidths=0.5, linecolor='white',
            cbar_kws={'label': 'Total Injured'}, ax=ax)
ax.set_title('Monthly Injuries Heatmap by Year', fontsize=14, fontweight='bold', pad=12)
ax.set_xlabel('Month')
ax.set_ylabel('Hijri Year')
ax.set_yticklabels(['1437 H', '1438 H', '1439 H'], rotation=0)
plt.tight_layout()
plt.show()


#### 💡 الاستنتاجات

- **رمضان (الشهر ٩)** يظهر بوضوح كأعلى شهر في الحوادث في جميع السنوات الثلاث.
- ارتفاع ملحوظ أيضاً في أشهر **رجب وشعبان** (٧ و٨)، فترة ما قبل رمضان.
- أشهر الشتاء (محرم وصفر) تسجّل بشكل عام أقل عدد من الحوادث.


### ٧.٣ الحوادث حسب المدينة — Horizontal Bar Chart

الأعمدة **الحمراء** = نسبة الوفيات أعلى من المتوسط.

In [ ]:
# --- حساب إجمالي المصابين والمتوفين لكل مدينة ---
city_inj = inj.groupby('city')[NUM_COLS].sum().sum(axis=1).sort_values()  # ترتيب تصاعدي للرسم
city_ded = ded.groupby('city')[NUM_COLS].sum().sum(axis=1)
fat_rate = city_ded / (city_inj + city_ded) * 100  # نسبة الوفيات لكل مدينة

# تلوين المدن بالأحمر إذا كانت نسبة وفياتها أعلى من المتوسط
colors = ['#C0392B' if fat_rate.get(c, 0) > fat_rate.median()
          else '#2980B9' for c in city_inj.index]

fig, ax = plt.subplots(figsize=(10, 7))
bars = ax.barh(city_inj.index, city_inj.values,
               color=colors, edgecolor='white', height=0.7)

# إضافة الأرقام بجانب كل شريط
for bar, v in zip(bars, city_inj.values):
    ax.text(v + 150, bar.get_y() + bar.get_height() / 2,
            f'{int(v):,}', va='center', fontsize=9)

ax.axvline(city_inj.median(), color='#2C3E50', lw=1.4, ls='--',
           label=f'Median  {int(city_inj.median()):,}')
ax.set_title('Total Injuries by City — 1437–1439 H\n(red = fatality rate above median)',
             fontsize=13, fontweight='bold', pad=12)
ax.set_xlabel('Total Injured Cases')
ax.legend(fontsize=10)
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.show()


#### 💡 الاستنتاجات

- **الرياض** تتصدر بفارق كبير في أعداد المصابين.
- **الباحة وتبوك** تسجّلان نسبة وفيات أعلى من المتوسط رغم إصاباتها المنخفضة — حوادثها **أشد فتكاً** (طرق جبلية وسريعة).
- **الطائف وجازان والقصيم** الأقل في نسبة الوفيات.


### ٧.٤ توزيع الفئات العمرية حسب المدينة — Stacked Bar Chart

نسبة كل فئة عمرية من المصابين في أكبر ١٢ مدينة.

In [ ]:
age_cols   = ['under_18', 'age_18_30', 'age_30_40', 'age_40_50', 'age_50_plus']
age_labels = ['< 18', '18–30', '30–40', '40–50', '50+']
AGE_COLORS = ['#1ABC9C', '#2980B9', '#8E44AD', '#E67E22', '#C0392B']

# اختيار أكبر 12 مدينة بالإصابات
top12 = (inj.groupby('city')[NUM_COLS].sum()
            .sum(axis=1).sort_values(ascending=False)
            .head(12).index)

city_age = inj.groupby('city')[age_cols].sum().loc[top12]
city_age.columns = age_labels

fig, ax = plt.subplots(figsize=(13, 6))
city_age.plot(kind='bar', stacked=True, color=AGE_COLORS,
              ax=ax, edgecolor='white', linewidth=0.5, width=0.75)
ax.set_title('Age Group Distribution by City (Top 12)', fontsize=13, fontweight='bold', pad=12)
ax.set_xlabel('City')
ax.set_ylabel('Total Injured')
ax.set_xticklabels(top12, rotation=35, ha='right', fontsize=9)
ax.legend(title='Age Group', bbox_to_anchor=(1.01, 1), loc='upper left', fontsize=9)
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.show()


#### 💡 الاستنتاجات

- فئة **١٨–٣٠ سنة** هي الأكثر تضرراً في جميع المدن دون استثناء.
- فئة **٣٠–٤٠ سنة** تأتي ثانياً — شريحة القوى العاملة الأكثر عرضة للخطر.


### ٧.٥ توزيع الجنس — Pie Chart

نسبة الذكور مقابل الإناث بين المصابين والمتوفين.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 5.5))

for ax, (dataset, title) in zip(axes, [(inj, 'Injured'), (ded, 'Deaths')]):
    m = dataset['male'].sum()    # إجمالي الذكور
    f = dataset['female'].sum()  # إجمالي الإناث
    wedges, texts, autotexts = ax.pie(
        [m, f], labels=['Male', 'Female'],
        colors=['#2980B9', '#E91E63'],
        autopct='%1.1f%%', startangle=90,
        wedgeprops={'edgecolor': 'white', 'linewidth': 2},
        textprops={'fontsize': 12}, explode=(0.04, 0.04)
    )
    for at in autotexts:
        at.set_fontsize(13); at.set_fontweight('bold')
    ax.set_title(f'Gender Split — {title}\n({int(m+f):,} total)',
                 fontsize=13, fontweight='bold', pad=14)

plt.suptitle('Gender Distribution: Injured vs Deaths', fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()


#### 💡 الاستنتاجات

- **الذكور** يمثّلون ~٨٣٪ من المصابين و~٨٧٪ من المتوفين.
- حملات السلامة المرورية يجب أن **تستهدف الذكور بشكل رئيسي**، وخاصة فئة ١٨–٣٠.


### ٧.٦ السعوديون مقابل غير السعوديين — Grouped Bar Chart

مقارنة الجنسيتين في أكبر ١٢ مدينة.

In [ ]:
city_saudi    = inj.groupby('city')['saudi'].sum().loc[top12]
city_nonsaudi = inj.groupby('city')['non_saudi'].sum().loc[top12]

x = np.arange(len(top12))
w = 0.38  # عرض كل شريط

fig, ax = plt.subplots(figsize=(13, 5.5))
b1 = ax.bar(x - w/2, city_saudi.values,    w, color='#1ABC9C', label='Saudi',     alpha=0.88, edgecolor='white')
b2 = ax.bar(x + w/2, city_nonsaudi.values, w, color='#8E44AD', label='Non-Saudi', alpha=0.88, edgecolor='white')

# إضافة الأرقام أعلى كل شريط
for b in [b1, b2]:
    for bar in b:
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 40,
                f'{int(bar.get_height()):,}', ha='center', va='bottom', fontsize=7.5)

ax.set_xticks(x)
ax.set_xticklabels(top12, rotation=35, ha='right', fontsize=9)
ax.set_title('Saudi vs Non-Saudi Injured — Top 12 Cities', fontsize=13, fontweight='bold', pad=12)
ax.set_ylabel('Total Injured')
ax.legend(fontsize=11)
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.show()


#### 💡 الاستنتاجات

- **مكة والمدينة وجدة** بها نسبة غير عادية من غير السعوديين بسبب موسم الحج والعمرة والعمالة الوافدة.
- في المدن الداخلية كحائل والقصيم، السعوديون يمثّلون الغالبية العظمى.


### ٧.٧ داخل المدينة مقابل خارجها — Box Plot

**Box Plot** يُظهر توزيع الأرقام:
- الصندوق = ٥٠٪ من البيانات الوسطى
- الخط في المنتصف = الوسيط
- النقاط خارج الخطوط = قيم شاذة (حوادث استثنائية)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5.5))
bp = ax.boxplot(
    [inj['inside_city'].dropna(), inj['outside_city'].dropna()],
    labels=['Inside City', 'Outside City'],
    patch_artist=True,  # تلوين الصناديق
    medianprops={'color': '#2C3E50', 'linewidth': 2.5},
    whiskerprops={'linewidth': 1.5},
    capprops={'linewidth': 1.5},
    flierprops={'marker': 'o', 'markersize': 4, 'alpha': 0.5}
)
for patch, col in zip(bp['boxes'], ['#2980B9', '#E67E22']):
    patch.set_facecolor(col)
    patch.set_alpha(0.75)

ax.set_title('Accident Distribution: Inside vs Outside City',
             fontsize=13, fontweight='bold', pad=12)
ax.set_ylabel('Cases per City-Month Record')
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.show()


#### 💡 الاستنتاجات

- حوادث **خارج المدينة** تُظهر تشتتاً أكبر وقيماً أعلى — **الطرق السريعة** أشد خطورة وأقل استقراراً.
- النقاط الشاذة في خارج المدينة تمثّل أشهراً استثنائية كرمضان وموسم الحج.


### ٧.٨ خريطة الارتباط — Correlation Heatmap

**Correlation:** يقيس قوة العلاقة بين متغيّرين (+١ = علاقة طردية، -١ = عكسية، ٠ = لا علاقة).

In [ ]:
corr = inj[NUM_COLS + ['month', 'year']].corr()  # حساب معاملات الارتباط

fig, ax = plt.subplots(figsize=(11, 8))
mask = np.triu(np.ones_like(corr, dtype=bool))  # إخفاء المثلث العلوي (مكرر)
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f',
            cmap='coolwarm', center=0, vmin=-1, vmax=1,
            linewidths=0.5, linecolor='white',
            cbar_kws={'shrink': 0.8}, ax=ax)
ax.set_title('Correlation Heatmap — Injured Dataset', fontsize=13, fontweight='bold', pad=12)
plt.tight_layout()
plt.show()


#### 💡 الاستنتاجات

- **ارتباط قوي جداً** بين الذكور والإجمالي الكلي — وهو متوقع بسبب هيمنتهم على الحوادث.
- **الشهر والسنة** لديهما ارتباط منخفض، مما يعني أن الأنماط الزمنية تحتاج تحليلاً أعمق.


### ٧.٩ الاتجاه الشهري والتنبؤ — Line Chart + Forecast

الحوادث الشهرية لكل سنة مع **خط تنبؤ** لسنة ١٤٤٠ هـ باستخدام **Linear Regression**.

In [ ]:
fig, ax = plt.subplots(figsize=(13, 5.5))
year_palette = {1437: '#2980B9', 1438: '#1ABC9C', 1439: '#8E44AD'}

# رسم الاتجاه الشهري لكل سنة
for yr in [1437, 1438, 1439]:
    mo = inj[inj['year'] == yr].groupby('month')[NUM_COLS].sum().sum(axis=1)
    ax.plot(mo.index, mo.values, 'o-', color=year_palette[yr], lw=2,
            ms=6, label=f'{yr} H', markerfacecolor='white', markeredgewidth=1.8)

# بناء نموذج Linear Regression للتنبؤ
inj_copy = inj.copy()
inj_copy['time_idx'] = (inj_copy['year'] - 1437) * 12 + inj_copy['month']  # رقم الشهر التسلسلي
inj_copy['period_total'] = inj_copy[NUM_COLS].sum(axis=1)
agg = inj_copy.groupby('time_idx')['period_total'].sum().reset_index()

lr = LinearRegression()
lr.fit(agg[['time_idx']], agg['period_total'])  # تدريب النموذج

# توقع شهور ١٤٤٠ هـ (37 إلى 48 تسلسلياً)
future_idx = np.arange(37, 49).reshape(-1, 1)
forecast   = lr.predict(future_idx)

ax.plot(range(1, 13), forecast, 'r--', lw=2, alpha=0.7, label='1440 H Forecast')
ax.fill_between(range(1, 13), forecast * 0.90, forecast * 1.10,
                color='red', alpha=0.07, label='±10% Error Band')

ax.axvspan(8.5, 9.5, alpha=0.1, color='#E67E22')  # تظليل شهر رمضان
ax.text(9.0, ax.get_ylim()[1] * 0.96, 'Ramadan', ha='center', fontsize=9, color='#E67E22')
ax.set_xticks(range(1, 13))
ax.set_xticklabels([m[:5] for m in MONTH_LABELS], rotation=30, fontsize=9)
ax.set_title('Monthly Injuries Trend & 1440 H Forecast', fontsize=13, fontweight='bold', pad=12)
ax.set_ylabel('Total Injured (All Cities)')
ax.legend(fontsize=9, ncol=2)
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.show()

slope = lr.coef_[0]
print(f'Trend slope: {slope:.1f} cases/month — Overall trend: {"increasing" if slope > 0 else "decreasing"}')


#### 💡 الاستنتاجات

- التوقع يشير إلى **استمرار الانخفاض التدريجي** في ١٤٤٠ هـ.
- **رمضان** يبقى الشهر الأعلى خطورة في كل سنة.
- نطاق الخطأ (±١٠٪) يُذكّر بأن هذا تنبؤ أولي يحتاج بيانات ١٤٤٠ للتحقق.


### ٧.١٠ تجميع المدن حسب الخطورة — Clustering Scatter Plot

كل نقطة = مدينة في شهر معين. **PCA** يُقلّص الأبعاد إلى بُعدين للرسم.

In [ ]:
# --- بناء جدول مدمج: مصابون + متوفون لكل مدينة-شهر-سنة ---
inj_g = inj.groupby(['city', 'year', 'month'])[NUM_COLS].sum().add_suffix('_inj').reset_index()
ded_g = ded.groupby(['city', 'year', 'month'])[NUM_COLS].sum().add_suffix('_ded').reset_index()
ml    = pd.merge(inj_g, ded_g, on=['city', 'year', 'month'])  # دمج على المفتاح المشترك

ml['total_injured'] = ml[[c for c in ml.columns if '_inj' in c]].sum(axis=1)
ml['total_dead']    = ml[[c for c in ml.columns if '_ded' in c]].sum(axis=1)
ml['fat_rate']      = ml['total_dead'] / (ml['total_injured'] + ml['total_dead'] + 1e-9)  # نسبة الوفيات

# --- تجهيز البيانات للـ K-Means ---
km_features = ['total_injured', 'total_dead', 'fat_rate',
               'outside_city_inj', 'outside_city_ded',
               'age_18_30_inj', 'age_30_40_inj']
X_km = StandardScaler().fit_transform(ml[km_features].fillna(0))  # تطبيع الأرقام

# --- تطبيق K-Means بـ 4 مجموعات ---
km = KMeans(n_clusters=4, random_state=42, n_init=10)
ml['cluster'] = km.fit_predict(X_km)

# --- تقليص الأبعاد إلى 2 للرسم ---
pca = PCA(n_components=2, random_state=42)
coords    = pca.fit_transform(X_km)
ml['pc1'] = coords[:, 0]
ml['pc2'] = coords[:, 1]

# إعادة ترقيم المجموعات تنازلياً حسب نسبة الوفيات (٠ = الأخطر)
mean_fat  = ml.groupby('cluster')['fat_rate'].mean().sort_values(ascending=False)
label_map = {old: new for new, old in enumerate(mean_fat.index)}
ml['cluster_label'] = ml['cluster'].map(label_map)

CLUSTER_COLORS = {0: '#C0392B', 1: '#E67E22', 2: '#2980B9', 3: '#1ABC9C'}

fig, ax = plt.subplots(figsize=(10, 7))
for c in sorted(ml['cluster_label'].unique()):
    sub = ml[ml['cluster_label'] == c]
    ax.scatter(sub['pc1'], sub['pc2'],
               color=CLUSTER_COLORS[c], alpha=0.65, s=50,
               edgecolors='white', lw=0.4,
               label=f'Cluster {c}  (fatality={sub["fat_rate"].mean()*100:.1f}%,  n={len(sub)})')

ax.set_title('City Risk Clustering — K-Means (k=4)', fontsize=13, fontweight='bold', pad=12)
ax.set_xlabel(f'PC1  ({pca.explained_variance_ratio_[0]*100:.1f}% of variance)')
ax.set_ylabel(f'PC2  ({pca.explained_variance_ratio_[1]*100:.1f}% of variance)')
ax.legend(fontsize=10)
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.show()

print('Cluster Summary:')
display(ml.groupby('cluster_label')[['total_injured', 'total_dead', 'fat_rate',
                                     'outside_city_inj']].mean().round(2))


#### 💡 الاستنتاجات

- **Cluster 0 (الأحمر):** نسبة وفيات مرتفعة مع إصابات قليلة — مدن صغيرة بحوادث طرق سريعة قاتلة.
- **Cluster 1 (البرتقالي):** حجم كبير من الوفيات — مدن رئيسية في أشهر الذروة.
- **Cluster 2 (الأزرق):** الأقل خطورة — مدن هادئة أو أشهر منخفضة الحوادث.
- **Cluster 3 (الأخضر):** خطورة متوسطة مرتبطة بالطرق الرئيسية.


## ٨. ملخص الاستنتاجات الرئيسية

| # | الاستنتاج | المخطط |
|---|---|---|
| ١ | انخفاض تدريجي في الحوادث من ١٤٣٧ إلى ١٤٣٩ | ٧.١ |
| ٢ | الذكور ~٨٣٪ من المصابين و~٨٧٪ من المتوفين | ٧.٥ |
| ٣ | فئة ١٨–٣٠ سنة الأكثر تضرراً في جميع المدن | ٧.٤ |
| ٤ | الرياض الأعلى في أعداد الإصابات الإجمالية | ٧.٣ |
| ٥ | حوادث خارج المدينة أشد تفاوتاً وأعلى خطورة | ٧.٧ |
| ٦ | رمضان يُسجّل أعلى ذروة في الحوادث سنوياً | ٧.٢ |
| ٧ | الباحة وتبوك الأعلى في نسبة الوفيات | ٧.٣ |
| ٨ | مكة والمدينة بهما نسبة غير سعوديين مرتفعة | ٧.٦ |
| ٩ | التنبؤ يشير لاستمرار الانخفاض في ١٤٤٠ هـ | ٧.٩ |
| ١٠ | ٤ مجموعات واضحة في أنماط خطورة المدن | ٧.١٠ |


## ٩. نموذج التعلم الآلي (ML Model)

### الهدف

نريد أن يتعلم الحاسوب من البيانات التاريخية، ثم يُجيب على هذا السؤال:

> **هل هذا السجل (مدينة + شهر + سنة) سيكون عالي الخطورة؟**

هذا يُسمى **Binary Classification** — المخرج إما `1` (خطر عالٍ) أو `0` (خطر منخفض).

---

### النماذج الثلاثة

| النموذج | الفكرة البسيطة |
|---|---|
| **Random Forest** | ٢٠٠ شجرة قرار تصوّت والأغلبية تحكم |
| **Gradient Boosting** | أشجار تتعلم بالتسلسل — كل واحدة تُصلح أخطاء السابقة |
| **Logistic Regression** | معادلة رياضية بسيطة تحسب احتمالية الخطر |

---

### مقياس الدقة — AUC

```
AUC = 1.0  →  نموذج مثالي
AUC = 0.9  →  ممتاز جداً
AUC = 0.7  →  مقبول
AUC = 0.5  →  تخمين عشوائي
```


In [ ]:
# --- الخطوة ١: تجهيز البيانات ---

le = LabelEncoder()
ml['city_enc']   = le.fit_transform(ml['city'])    # تحويل اسم المدينة إلى رقم
ml['is_ramadan'] = (ml['month'] == 9).astype(int)  # ١ إذا كان الشهر رمضان، ٠ غيره

# تصنيف السجلات: أعلى 33% نسبة وفيات = high_risk
threshold       = ml['fat_rate'].quantile(0.67)
ml['high_risk'] = (ml['fat_rate'] >= threshold).astype(int)

print(f'Fatality threshold : {threshold:.3f} ({threshold*100:.1f}%)')
print(f'High risk (1)      : {ml["high_risk"].sum()}')
print(f'Low risk  (0)      : {(ml["high_risk"]==0).sum()}')

# --- الخطوة ٢: تحديد المدخلات (Features) ---
FEATURES = [
    'city_enc',          # المدينة
    'month',             # الشهر
    'year',              # السنة
    'is_ramadan',        # هل رمضان؟
    'male_inj',          # ذكور مصابون
    'female_inj',        # إناث مصابات
    'inside_city_inj',   # حوادث داخل المدينة
    'outside_city_inj',  # حوادث خارج المدينة
    'under_18_inj',      # مصابون أقل من 18
    'age_18_30_inj',     # مصابون 18–30
    'age_30_40_inj',     # مصابون 30–40
    'age_40_50_inj',     # مصابون 40–50
    'age_50_plus_inj',   # مصابون 50+
    'saudi_inj',         # مصابون سعوديون
    'non_saudi_inj',     # مصابون غير سعوديين
    'outside_city_ded',  # وفيات خارج المدينة
    'age_18_30_ded',     # وفيات 18–30
    'age_30_40_ded',     # وفيات 30–40
]

X = ml[FEATURES].fillna(0)
y = ml['high_risk']

# --- الخطوة ٣: تقسيم 75% تدريب / 25% اختبار ---
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y)  # stratify: نسبة متساوية في كلا القسمين

# تطبيع الأرقام (ضروري لـ Logistic Regression)
scaler = StandardScaler()
X_tr_s = scaler.fit_transform(X_train)
X_te_s = scaler.transform(X_test)

print(f'\nTraining size : {X_train.shape}')
print(f'Testing size  : {X_test.shape}')


In [ ]:
# --- الخطوة ٤: تدريب النماذج وقياس الدقة ---

MODELS = {
    'Random Forest':       RandomForestClassifier(n_estimators=200, max_depth=6, random_state=42),
    'Gradient Boosting':   GradientBoostingClassifier(n_estimators=150, max_depth=4, random_state=42),
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
}
MODEL_COLORS = ['#2980B9', '#1ABC9C', '#8E44AD']

results = {}
for name, model in MODELS.items():
    scaled = (name == 'Logistic Regression')         # Logistic Regression يحتاج بيانات مطبّعة
    Xtr, Xte = (X_tr_s, X_te_s) if scaled else (X_train, X_test)
    model.fit(Xtr, y_train)                           # تدريب النموذج
    preds = model.predict(Xte)                        # التوقع على بيانات الاختبار
    proba = model.predict_proba(Xte)[:, 1]            # احتمالية أن يكون high_risk=1
    cv    = cross_val_score(model, Xtr, y_train, cv=5, scoring='roc_auc')  # 5-Fold CV
    fpr, tpr, _ = roc_curve(y_test, proba)
    results[name] = {
        'model': model, 'preds': preds, 'proba': proba,
        'auc': roc_auc_score(y_test, proba),
        'cm':  confusion_matrix(y_test, preds),
        'fpr': fpr, 'tpr': tpr, 'cv': cv
    }
    print(f'{name:<25}  AUC = {results[name]["auc"]:.3f}  |  CV = {cv.mean():.3f} ± {cv.std():.3f}')


In [ ]:
# --- رسم نتائج النماذج الثلاثة ---

fig, axes = plt.subplots(1, 3, figsize=(17, 5))

# المخطط ١: ROC Curve — كلما اقترب من الزاوية العلوية اليسرى كان أدق
ax = axes[0]
for (name, res), col in zip(results.items(), MODEL_COLORS):
    ax.plot(res['fpr'], res['tpr'], lw=2, color=col,
            label=f'{name}\nAUC={res["auc"]:.3f}')
ax.plot([0, 1], [0, 1], 'k--', alpha=0.4, label='Random Guess')
ax.fill_between(results['Random Forest']['fpr'],
                results['Random Forest']['tpr'], alpha=0.06, color='#2980B9')
ax.set(xlabel='False Positive Rate', ylabel='True Positive Rate',
       title='ROC Curves — All Models')
ax.legend(fontsize=8, loc='lower right')
ax.spines[['top', 'right']].set_visible(False)

# المخطط ٢: مقارنة AUC عبر 5-Fold Cross Validation
ax = axes[1]
cv_means = [results[n]['cv'].mean() for n in MODELS]
cv_stds  = [results[n]['cv'].std()  for n in MODELS]
bars = ax.bar(list(MODELS.keys()), cv_means, color=MODEL_COLORS,
              alpha=0.85, edgecolor='white', yerr=cv_stds, capsize=5)
for b, v in zip(bars, cv_means):
    ax.text(b.get_x() + b.get_width() / 2, v + 0.005,
            f'{v:.3f}', ha='center', fontsize=10, fontweight='bold')
ax.set_ylim(0.5, 1.05)
ax.set_ylabel('Mean AUC (5-Fold CV)')
ax.set_title('Cross-Validation AUC Comparison')
ax.set_xticklabels(list(MODELS.keys()), rotation=12, fontsize=9)
ax.spines[['top', 'right']].set_visible(False)

# المخطط ٣: Confusion Matrix لأفضل نموذج
ax = axes[2]
best = max(results, key=lambda n: results[n]['auc'])  # النموذج بأعلى AUC
disp = ConfusionMatrixDisplay(results[best]['cm'],
                               display_labels=['Low Risk', 'High Risk'])
disp.plot(ax=ax, colorbar=False, cmap='Blues')
ax.set_title(f'Confusion Matrix\n({best})')

plt.suptitle('ML Model Evaluation', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print(f'Best model: {best}  AUC = {results[best]["auc"]:.3f}')


#### 💡 استنتاجات نماذج ML

- **Logistic Regression** حقّق أعلى دقة (AUC ≈ 0.965) رغم بساطته — مما يدل على أن العلاقة بين المتغيرات **خطية نسبياً**.
- جميع النماذج تجاوزت **٠.٨٨** — وهي نتيجة ممتازة.
- الفارق الضيّق بين النتائج يعني أن البيانات **نظيفة ومنتظمة** وتسمح بتوقعات موثوقة.


In [ ]:
# --- أهمية المتغيرات في نموذج Random Forest ---

rf = results['Random Forest']['model']
fi = pd.Series(rf.feature_importances_, index=FEATURES).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(9, 6))
# تلوين المتغيرات المتعلقة بالوفيات/خارج المدينة باللون الأحمر
colors_fi = ['#C0392B' if ('ded' in i or 'outside' in i) else '#2980B9' for i in fi.index]
ax.barh(fi.index, fi.values, color=colors_fi, edgecolor='white', alpha=0.87)
ax.set_title('Feature Importances — Random Forest\n(red = death/outside-city related)',
             fontsize=13, fontweight='bold', pad=12)
ax.set_xlabel('Importance Score')
for i, v in enumerate(fi.values):
    ax.text(v + 0.001, i, f'{v:.3f}', va='center', fontsize=8.5)
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.show()


#### 💡 ماذا يعني هذا؟

- **وفيات خارج المدينة** هو المتغير الأكثر تأثيراً — الطرق السريعة هي مصدر الخطر الأول.
- **وفيات الفئة ١٨–٣٠** تأتي ثانياً — الشباب الأكثر عرضة للحوادث المميتة.
- **هوية المدينة** مهمة — لكل مدينة طابع خاص لا يرتبط فقط بالأرقام.


## ١٠. نموذج التجميع (Clustering Model)

### الفرق بين ML و Clustering

| ML (القسم ٩) | Clustering (هذا القسم) |
|---|---|
| إجابة مسبقة (٠ أو ١) | لا توجد إجابة مسبقة |
| **Supervised** — يتعلم من أمثلة | **Unsupervised** — يكتشف الأنماط بنفسه |
| السؤال: هل هذا السجل خطر؟ | السؤال: ما المجموعات الطبيعية في البيانات؟ |

### اختيار عدد المجموعات — Elbow Method

نجرّب قيماً مختلفة لـ k ونختار نقطة 'الكوع' حيث تبدأ الفائدة بالتناقص.

In [ ]:
# --- Elbow Method لتحديد أمثل عدد للمجموعات ---

inertias = []  # Inertia = مجموع المسافات بين كل نقطة ومركز مجموعتها
K_range  = range(2, 10)
for k in K_range:
    km_e = KMeans(n_clusters=k, random_state=42, n_init=10)
    km_e.fit(X_km)
    inertias.append(km_e.inertia_)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(K_range, inertias, 'o-', color='#2980B9', lw=2.2,
        ms=7, markerfacecolor='white', markeredgewidth=2)
ax.axvline(4, color='#C0392B', lw=1.5, ls='--', label='Chosen k=4')
ax.set(xlabel='Number of Clusters (k)', ylabel='Inertia',
       title='Elbow Method — Optimal k Selection')
ax.legend(fontsize=10)
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.show()


#### 💡 كيف تقرأ هذا المخطط؟

- عند **k=4** يظهر 'كوع' واضح — زيادة عدد المجموعات بعده لا تُحسّن النتائج كثيراً.
- لهذا اخترنا k=4 كأمثل عدد للمجموعات.


In [ ]:
# --- ملخص خصائص كل مجموعة ---

summary = (ml.groupby('cluster_label')
             [['total_injured', 'total_dead', 'fat_rate', 'outside_city_inj']]
             .mean().round(2))
summary.index   = [f'Cluster {i}' for i in summary.index]
summary.columns = ['Avg Injured', 'Avg Deaths', 'Fatality Rate', 'Avg Outside Injured']
summary['Fatality Rate'] = (summary['Fatality Rate'] * 100).round(1).astype(str) + '%'
print('Cluster Profiles:')
display(summary)


In [ ]:
# --- مخطط التجميع النهائي ---

CLUSTER_COLORS = {0: '#C0392B', 1: '#E67E22', 2: '#2980B9', 3: '#1ABC9C'}

fig, ax = plt.subplots(figsize=(10, 7))
for c in sorted(ml['cluster_label'].unique()):
    sub = ml[ml['cluster_label'] == c]
    ax.scatter(sub['pc1'], sub['pc2'],
               color=CLUSTER_COLORS[c], alpha=0.65, s=50,
               edgecolors='white', lw=0.4,
               label=f'Cluster {c}  (fatality={sub["fat_rate"].mean()*100:.1f}%,  n={len(sub)})')

ax.set_title('City Risk Clustering — K-Means (k=4) PCA View',
             fontsize=13, fontweight='bold', pad=12)
ax.set_xlabel(f'PC1  ({pca.explained_variance_ratio_[0]*100:.1f}% of variance)')
ax.set_ylabel(f'PC2  ({pca.explained_variance_ratio_[1]*100:.1f}% of variance)')
ax.legend(fontsize=10)
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.show()


#### 💡 الاستنتاجات والتوصيات حسب المجموعة

| المجموعة | الوصف | التوصية |
|---|---|---|
| Cluster 0 🔴 | نسبة وفيات عالية، إصابات قليلة | وحدات إسعاف على الطرق السريعة |
| Cluster 1 🟠 | حجم وفيات كبير، مدن رئيسية | حملات توعية + زيادة دوريات الذروة |
| Cluster 2 🔵 | الأقل خطورة | مراقبة روتينية كافية |
| Cluster 3 🟢 | خطورة متوسطة | تحسين إشارات الطرق الرئيسية |


## ١١. الخلاصة والتوصيات

### ملخص النتائج

**الأنماط الزمنية:** انخفاض تدريجي يدل على تحسّن في السلامة، ورمضان يبقى الشهر الأخطر سنوياً.

**المخاطر الديموغرافية:** الذكور في الفئة ١٨–٣٠ الأكثر تضرراً. مكة والمدينة وجدة تعكس تنوعاً بالجنسيات.

**المخاطر الجغرافية:** الرياض الأعلى في الإصابات. الباحة وتبوك الأعلى في معدل الوفيات. الحوادث خارج المدن أشد فتكاً.

**نتائج الذكاء الاصطناعي:** جميع النماذج حققت AUC > 0.88. وفيات خارج المدينة والفئة ١٨–٣٠ أقوى المتنبئات. K-Means كشف ٤ مجموعات واضحة.

---

### التوصيات

| الأولوية | الإجراء المقترح | الهدف |
|---|---|---|
| 🔴 عاجل | نشر وحدات إسعاف على الطرق السريعة | الباحة، تبوك، الطرق الجبلية |
| 🔴 عاجل | تشديد المراقبة في رمضان | جميع المدن |
| 🟠 متوسط | حملات توعية للذكور ١٨–٣٠ | جميع المدن |
| 🟠 متوسط | موارد السلامة متعددة اللغات | مكة، المدينة، جدة |
| 🟡 مستقبلي | تحديث النموذج ببيانات ١٤٤٠ هـ | جميع المناطق |
